# Multi-Project Pipeline Runner

Run the PAD2Skills pipeline **step-by-step across multiple projects**: for each stage (e.g., PDF → Markdown, sectioning, summarization, extraction, matching), you run that stage for *all selected projects* before moving on to the next stage.

This is intentionally different from the **single-project pipeline**, which will run *all stages end-to-end for one project* before starting the next.

## 0. Setup

### 0.01 Import Required Libraries

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd

# Import our config
import sys
sys.path.append(str(Path.cwd().parent))
from src.config import load_config

### 0.02 Load Configuration and Environment Variables

In [2]:
# Load environment variables from .env file
project_root = Path.cwd().parent
env_path = project_root / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        f"'.env' file not found at {env_path}\n"
        "Please copy .env.example to .env and add your OpenAI API key."
    )

# Load from specific path
load_dotenv(env_path, override=True)

# Load project config
config = load_config()

# Get OpenAI API key from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Verify API key is set
if not OPENAI_API_KEY:
    raise ValueError("Missing required environment variable: OPENAI_API_KEY")

print("✓ Environment variables loaded")
print(f"  API Key: {OPENAI_API_KEY[:10]}...{OPENAI_API_KEY[-4:]}")

✓ Environment variables loaded
  API Key: sk-proj-cj...__0A


### 0.03 Set Up Paths

In [3]:
# Get paths
project_data_dir = project_root / "data" / "bronze" / "project_data"
all_project_details_path = project_data_dir / "all_project_details.csv"
project_summary_path = project_data_dir / "project_summary.csv"

print(f"Project data directory: {project_data_dir}")
print(f"All project details path: {all_project_details_path}")
print(f"Project summary path: {project_summary_path}")
print(f"Files exist: {all_project_details_path.exists()} / {project_summary_path.exists()}")

Project data directory: /Users/lauren/repos/PAD2Skills/data/bronze/project_data
All project details path: /Users/lauren/repos/PAD2Skills/data/bronze/project_data/all_project_details.csv
Project summary path: /Users/lauren/repos/PAD2Skills/data/bronze/project_data/project_summary.csv
Files exist: True / True


### 0.04 Set Overwrite Sections

In [4]:
# Useful to set file overwrite flags at the top of the script
# ow = overwrite
ow_3_03_convert_pdf = False
ow_4_03_extract_sections = False
ow_5_03_extract_abbr = False
ow_6_03_create_md_chunks = False
ow_7_03_long_summary = False
ow_7_04_short_summary = False
# Important overwrites
ow_8_03_extract_occupations = False
ow_8_04_extract_occs_csv = False
ow_9_02_esco_embeddings = False
ow_9_04_esco_matching = False
ow_10_03_esco_selection = False
ow_10_06_unique_esco = False
ow_11_05_match_nace = False
ow_12_03_refine_skills = False
ow_13_03_create_crosswalk = False
ow_13_05_merge_onet = False

## 1. Get Project Details

### 1.01 Load Project Data

In [5]:
# Read project details and summary
all_project_details = pd.read_csv(all_project_details_path)
project_summary = pd.read_csv(project_summary_path)

print(f"All project details: {all_project_details.shape[0]} rows, {all_project_details.shape[1]} columns")
print(f"Project summary: {project_summary.shape[0]} rows, {project_summary.shape[1]} columns")

All project details: 123 rows, 20 columns
Project summary: 123 rows, 4 columns


### 1.02 Convert column names to snake_case

In [6]:
# Convert column names to snake_case
all_project_details.columns = (
    all_project_details.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w]', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)

print("Column names after standardization:")
print(list(all_project_details.columns))

Column names after standardization:
['project_id', 'status', 'team_leader', 'borrower_2', 'country', 'disclosure_date', 'approval_date', 'effective_date', 'total_project_cost_1', 'implementing_agency', 'region', 'fiscal_year_3', 'commitment_amount', 'environmental_category', 'environmental_and_social_risk', 'closing_date', 'last_stage_reached', 'last_update_date', 'consultant_services_required', 'associated_projects']


### 1.03 Merge and Filter Projects

In [7]:
# Merge project_summary to all_project_details
all_projects = all_project_details.merge(
    project_summary,
    on="project_id",
    how="left"
)

print(f"Merged data: {all_projects.shape[0]} rows, {all_projects.shape[1]} columns")

# Keep only projects with downloaded PADs
all_projects = all_projects[all_projects["pads_downloaded"] >= 1]

print(f"Projects with downloaded PADs: {all_projects.shape[0]} rows")
print(f"\nFirst few projects:")
print(all_projects.head())

Merged data: 123 rows, 23 columns
Projects with downloaded PADs: 98 rows

First few projects:
  project_id  status                                        team_leader  \
1    P119893  Closed                    Abdulhakim Mohammed Abdisubhan    
3    P173506  Active  Didier Makoso Tsasa , Fabrice Karl Bertholet, ...   
4    P176731  Active  Janina Franco , Abdulhakim Mohammed Abdisubhan...   
5    P507759  Active                     Jenny Jing Chao , Maria Arango   
6    P180547  Active   Monali Ranade , Dana Rysankova, Alona Kazantseva   

                                          borrower_2  \
1            Federal Democratic Republic of Ethiopia   
3                       DEMOCRATIC REPUBLIC OF CONGO   
4            Federal Democratic Republic of Ethiopia   
5                             Republic of Mozambique   
6  Common Market for Eastern and Southern Africa ...   

                         country    disclosure_date  \
1                       Ethiopia  December 22, 2011   
3  Congo

## 2. Select Projects

### 2.01 Select first 10 Projects or define your own list

In [8]:
# All projects
selected_projects = all_projects["project_id"][0:50].tolist()

print(f"Selected {len(selected_projects)} projects")
#for project_id in selected_projects:
#    print(f"  {project_id}")

Selected 50 projects


In [9]:
# Custom list (10 plus important project)
#selected_projects = ['P119893', 'P173506', 'P176731', 'P507759', 'P180547', 'P505856', 'P181341', 'P075941', 'P160708', 'P153743', 'P511453']
# selected_projects = ['P511453']
# selected_projects = ['P075941', 'P160708', 'P153743', 'P511453']


### 2.02 Filter Projects DataFrame

In [10]:
# Filter projects dataframe for selected projects
projects_df = all_projects[all_projects["project_id"].isin(selected_projects)]

print(f"Filtered to {len(projects_df)} projects:")
print(projects_df[["project_id", "status", "country"]].to_string(index=False))

Filtered to 50 projects:
project_id status                       country
   P119893 Closed                      Ethiopia
   P173506 Active Congo, Democratic Republic of
   P176731 Active                      Ethiopia
   P507759 Active                    Mozambique
   P180547 Active   Eastern and Southern Africa
   P505856 Active                    Seychelles
   P181341 Active  Somalia, Federal Republic of
   P075941 Closed   Eastern and Southern Africa
   P160708 Active    Western and Central Africa
   P153743 Closed                         Niger
   P166796 Active                          Mali
   P175295 Active                    Mozambique
   P144135 Closed                         Gabon
   P164225 Closed                        Guinea
   P179797 Active                    Mozambique
   P164044 Active    Western and Central Africa
   P503941 Active                        Zambia
   P179380 Active                        Zambia
   P167569 Active    Western and Central Africa
   P168185 Acti

### 2.03 Save Selected Projects Data

In [11]:
# Save filtered projects to silver directory
output_dir = project_root / "data" / "silver" / "selected_projects_data"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "selected_projects.csv"
projects_df.to_csv(output_path, index=False)

print(f"Saved {len(projects_df)} projects to:")
print(f"  {output_path}")

Saved 50 projects to:
  /Users/lauren/repos/PAD2Skills/data/silver/selected_projects_data/selected_projects.csv


## 3. Pipeline Step #1: Convert PDFs to Markdown

### 3.01 Import PDF Conversion Module

In [12]:
from src.pdf_conversion.converter import convert_pdfs

print("✓ PDF conversion module imported")

/Users/lauren/repos/PAD2Skills/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ PDF conversion module imported


### 3.02 Set Up PDF Conversion Paths

In [13]:
# Set up paths for PDF conversion
pdf_dir = project_root / config.paths.raw_pdfs
markdown_dir = project_root / config.paths.markdown

print(f"PDF directory: {pdf_dir}")
print(f"Markdown directory: {markdown_dir}")
print(f"PDF directory exists: {pdf_dir.exists()}")

PDF directory: /Users/lauren/repos/PAD2Skills/data/bronze/pads_pdf
Markdown directory: /Users/lauren/repos/PAD2Skills/data/silver/pads_md
PDF directory exists: True


### 3.03 Convert PDFs for Selected Projects

In [14]:
# Convert PDFs for each selected project
for project_id in selected_projects:
    pdf_file = pdf_dir / f"{project_id}_1.pdf"
    
    # Skip if PDF doesn't exist
    if not pdf_file.exists():
        print(f"⚠ PDF not found: {project_id}")
        continue
    
    # Convert single PDF using the src utility
    results = convert_pdfs(
        pdf_dir=pdf_dir,
        output_dir=markdown_dir,
        specific_pdf=pdf_file.name,
        overwrite=ow_3_03_convert_pdf,
        accurate_tables=True
    )
    
    # Report result
    if results["converted"]:
        print(f"✓ Converted: {project_id}")
    elif results["skipped"]:
        print(f"○ Skipped (already exists): {project_id}")
    elif results["failed"]:
        error = results["failed"][0][1]
        print(f"✗ Failed: {project_id} - {error}")

○ Skipped (already exists): P119893
○ Skipped (already exists): P173506
○ Skipped (already exists): P176731
○ Skipped (already exists): P507759
○ Skipped (already exists): P180547
○ Skipped (already exists): P505856
○ Skipped (already exists): P181341
○ Skipped (already exists): P075941
○ Skipped (already exists): P160708
○ Skipped (already exists): P153743
○ Skipped (already exists): P166796
○ Skipped (already exists): P175295
○ Skipped (already exists): P144135
○ Skipped (already exists): P164225
○ Skipped (already exists): P179797
○ Skipped (already exists): P164044
○ Skipped (already exists): P503941
○ Skipped (already exists): P179380
○ Skipped (already exists): P167569
○ Skipped (already exists): P168185
○ Skipped (already exists): P171742
○ Skipped (already exists): P166170
○ Skipped (already exists): P164885
○ Skipped (already exists): P180575
○ Skipped (already exists): P178914
○ Skipped (already exists): P173749
○ Skipped (already exists): P120304
○ Skipped (already exists): 

## 4. Pipeline Step #2: Extract Document Sections

### 4.01 Import Section Extraction Module

In [15]:
from src.extraction.extractor import extract_all_sections

print("✓ Section extraction module imported")

✓ Section extraction module imported


### 4.02 Set Up Section Extraction Paths

In [16]:
# Set up paths for section extraction
sections_output_dir = project_root / "data" / "silver" / "document_sections"

print(f"Markdown directory: {markdown_dir}")
print(f"Sections output directory: {sections_output_dir}")

Markdown directory: /Users/lauren/repos/PAD2Skills/data/silver/pads_md
Sections output directory: /Users/lauren/repos/PAD2Skills/data/silver/document_sections


### 4.03 Extract Sections for Selected Projects

In [17]:
# Extract sections for each selected project
for project_id in selected_projects:
    md_file = markdown_dir / f"{project_id}_1.md"
    
    # Skip if markdown doesn't exist
    if not md_file.exists():
        print(f"⚠ Markdown not found: {project_id}")
        continue
    
    # Extract sections using the src utility
    results = extract_all_sections(
        markdown_dir=markdown_dir,
        output_dir=sections_output_dir,
        specific_file=md_file.name,
        overwrite=ow_4_03_extract_sections
    )
    
    # Report result
    if results["extracted"]:
        print(f"✓ Extracted sections: {project_id}")
    elif results["skipped"]:
        print(f"○ Skipped (already exists): {project_id}")
    elif results["failed"]:
        error = results["failed"][0][1]
        print(f"✗ Failed: {project_id} - {error}")

○ Skipped (already exists): P119893
○ Skipped (already exists): P173506
○ Skipped (already exists): P176731
○ Skipped (already exists): P507759
○ Skipped (already exists): P180547
○ Skipped (already exists): P505856
○ Skipped (already exists): P181341
○ Skipped (already exists): P075941
○ Skipped (already exists): P160708
○ Skipped (already exists): P153743
○ Skipped (already exists): P166796
○ Skipped (already exists): P175295
○ Skipped (already exists): P144135
○ Skipped (already exists): P164225
○ Skipped (already exists): P179797
○ Skipped (already exists): P164044
○ Skipped (already exists): P503941
○ Skipped (already exists): P179380
○ Skipped (already exists): P167569
○ Skipped (already exists): P168185
○ Skipped (already exists): P171742
○ Skipped (already exists): P166170
○ Skipped (already exists): P164885
○ Skipped (already exists): P180575
○ Skipped (already exists): P178914
○ Skipped (already exists): P173749
○ Skipped (already exists): P120304
○ Skipped (already exists): 

## 5. Pipeline Step #3: Extract Abbreviations

### 5.01 Import Abbreviation Extraction Module

In [18]:
from src.extraction.extractor import extract_all_abbreviations

print("✓ Abbreviation extraction module imported")

✓ Abbreviation extraction module imported


### 5.02 Set Up Abbreviation Extraction Paths

In [19]:
# Set up paths for abbreviation extraction
abbreviations_output_dir = project_root / "data" / "silver" / "abbreviations_md"

print(f"Markdown directory: {markdown_dir}")
print(f"Abbreviations output directory: {abbreviations_output_dir}")

Markdown directory: /Users/lauren/repos/PAD2Skills/data/silver/pads_md
Abbreviations output directory: /Users/lauren/repos/PAD2Skills/data/silver/abbreviations_md


### 5.03 Extract Abbreviations for Selected Projects

In [20]:
# Extract abbreviations for each selected project
for project_id in selected_projects:
    md_file = markdown_dir / f"{project_id}_1.md"
    
    # Skip if markdown doesn't exist
    if not md_file.exists():
        print(f"⚠ Markdown not found: {project_id}")
        continue
    
    # Extract abbreviations using the src utility
    results = extract_all_abbreviations(
        markdown_dir=markdown_dir,
        output_dir=abbreviations_output_dir,
        specific_file=md_file.name,
        overwrite=ow_5_03_extract_abbr
    )
    
    # Report result
    if results["extracted"]:
        print(f"✓ Extracted abbreviations: {project_id}")
    elif results["skipped"]:
        print(f"○ Skipped (already exists): {project_id}")
    elif results["failed"]:
        error = results["failed"][0][1]
        print(f"✗ Failed: {project_id} - {error}")

○ Skipped (already exists): P119893
○ Skipped (already exists): P173506
○ Skipped (already exists): P176731
○ Skipped (already exists): P507759
○ Skipped (already exists): P180547
○ Skipped (already exists): P505856
○ Skipped (already exists): P181341
○ Skipped (already exists): P075941
○ Skipped (already exists): P160708
○ Skipped (already exists): P153743
○ Skipped (already exists): P166796
○ Skipped (already exists): P175295
○ Skipped (already exists): P144135
○ Skipped (already exists): P164225
○ Skipped (already exists): P179797
○ Skipped (already exists): P164044
○ Skipped (already exists): P503941
○ Skipped (already exists): P179380
○ Skipped (already exists): P167569
○ Skipped (already exists): P168185
○ Skipped (already exists): P171742
○ Skipped (already exists): P166170
○ Skipped (already exists): P164885
○ Skipped (already exists): P180575
○ Skipped (already exists): P178914
○ Skipped (already exists): P173749
○ Skipped (already exists): P120304
○ Skipped (already exists): 

## 6. Pipeline Step #4: Create Chunked Markdown Files

### 6.01 Import Chunking Module

In [21]:
from src.extraction.extractor import create_chunks

print("✓ Chunking module imported")

✓ Chunking module imported


### 6.02 Set Up Chunking Paths

In [22]:
# Set up paths for chunking
chunks_output_dir = project_root / "data" / "silver" / "pads_md_chunks"

print(f"Markdown directory: {markdown_dir}")
print(f"Sections directory: {sections_output_dir}")
print(f"Chunks output directory: {chunks_output_dir}")

Markdown directory: /Users/lauren/repos/PAD2Skills/data/silver/pads_md
Sections directory: /Users/lauren/repos/PAD2Skills/data/silver/document_sections
Chunks output directory: /Users/lauren/repos/PAD2Skills/data/silver/pads_md_chunks


### 6.03 Create Chunks for Selected Projects

In [23]:
# Create chunks for each selected project
for project_id in selected_projects:
    md_file = markdown_dir / f"{project_id}_1.md"
    
    # Skip if markdown doesn't exist
    if not md_file.exists():
        print(f"⚠ Markdown not found: {project_id}")
        continue
    
    # Create chunks using the src utility
    results = create_chunks(
        markdown_dir=markdown_dir,
        sections_dir=sections_output_dir,
        output_dir=chunks_output_dir,
        specific_file=md_file.name,
        overwrite=ow_6_03_create_md_chunks
    )
    
    # Report result
    if results["chunked"]:
        # Count chunks created for this project
        chunk_files = list(chunks_output_dir.glob(f"{project_id}_*.md"))
        print(f"✓ Created chunks: {project_id} ({len(chunk_files)} chunks)")
    elif results["skipped"]:
        print(f"○ Skipped (no sections or already exists): {project_id}")
    elif results["failed"]:
        error = results["failed"][0][1]
        print(f"✗ Failed: {project_id} - {error}")

○ Skipped (no sections or already exists): P119893
○ Skipped (no sections or already exists): P173506
○ Skipped (no sections or already exists): P176731
○ Skipped (no sections or already exists): P507759


○ Skipped (no sections or already exists): P180547
○ Skipped (no sections or already exists): P505856
○ Skipped (no sections or already exists): P181341
○ Skipped (no sections or already exists): P075941
○ Skipped (no sections or already exists): P160708
○ Skipped (no sections or already exists): P153743
○ Skipped (no sections or already exists): P166796
○ Skipped (no sections or already exists): P175295
○ Skipped (no sections or already exists): P144135
○ Skipped (no sections or already exists): P164225
○ Skipped (no sections or already exists): P179797
○ Skipped (no sections or already exists): P164044
○ Skipped (no sections or already exists): P503941
○ Skipped (no sections or already exists): P179380
○ Skipped (no sections or already exists): P167569
○ Skipped (no sections or already exists): P168185
○ Skipped (no sections or already exists): P171742
○ Skipped (no sections or already exists): P166170
○ Skipped (no sections or already exists): P164885
○ Skipped (no sections or alrea

## 7. Pipeline Step #5: Generate PAD Summaries

### 7.01 Import Summary Generation Module

In [24]:
from src.extraction.summarizer import generate_all_summaries

print("✓ Summary generation module imported")

✓ Summary generation module imported


### 7.02 Set Up Summary Generation Paths

In [25]:
# Set up paths for summary generation
summaries_output_dir = project_root / "data" / "silver" / "pad_summaries"

print(f"Chunks directory: {chunks_output_dir}")
print(f"Abbreviations directory: {abbreviations_output_dir}")
print(f"Summaries output directory: {summaries_output_dir}")

Chunks directory: /Users/lauren/repos/PAD2Skills/data/silver/pads_md_chunks
Abbreviations directory: /Users/lauren/repos/PAD2Skills/data/silver/abbreviations_md
Summaries output directory: /Users/lauren/repos/PAD2Skills/data/silver/pad_summaries


### 7.03 Generate Summaries for Selected Projects

In [26]:
# Generate summaries for each selected project
for project_id in selected_projects:
    # Check if chunks exist for this project
    chunk_files = list(chunks_output_dir.glob(f"{project_id}_*.md"))
    
    if not chunk_files:
        print(f"⚠ No chunks found: {project_id}")
        continue
    
    # Generate summary using the src utility
    results = generate_all_summaries(
        chunks_dir=chunks_output_dir,
        output_dir=summaries_output_dir,
        abbr_dir=abbreviations_output_dir,
        specific_project=project_id,
        num_chunks=4,
        overwrite=ow_7_03_long_summary
    )
    
    # Report result
    if results["generated"]:
        summary_file = summaries_output_dir / f"{project_id}_summary.txt"
        summary_text = summary_file.read_text(encoding="utf-8")
        word_count = len(summary_text.split())
        print(f"✓ Generated summary: {project_id} ({word_count} words)")
    elif results["skipped"]:
        print(f"○ Skipped (already exists): {project_id}")
    elif results["failed"]:
        error = results["failed"][0][1]
        print(f"✗ Failed: {project_id} - {error}")

○ Skipped (already exists): P119893


○ Skipped (already exists): P173506
○ Skipped (already exists): P176731
○ Skipped (already exists): P507759
○ Skipped (already exists): P180547
○ Skipped (already exists): P505856
○ Skipped (already exists): P181341
○ Skipped (already exists): P075941
○ Skipped (already exists): P160708
○ Skipped (already exists): P153743
○ Skipped (already exists): P166796
○ Skipped (already exists): P175295
○ Skipped (already exists): P144135
○ Skipped (already exists): P164225
○ Skipped (already exists): P179797
○ Skipped (already exists): P164044
○ Skipped (already exists): P503941
○ Skipped (already exists): P179380
○ Skipped (already exists): P167569
○ Skipped (already exists): P168185
○ Skipped (already exists): P171742
○ Skipped (already exists): P166170
○ Skipped (already exists): P164885
○ Skipped (already exists): P180575
○ Skipped (already exists): P178914
○ Skipped (already exists): P173749
○ Skipped (already exists): P120304
○ Skipped (already exists): P176620
○ Skipped (already exists): 

### 7.04 Generate Short Summaries for Selected Projects

In [27]:
from src.extraction.short_summarizer import generate_all_short_summaries

# Set up paths for short summary generation
short_summary_output_dir = project_root / "data" / "silver" / "short_summary_json"

print(f"Summaries directory: {summaries_output_dir}")
print(f"Short summary output directory: {short_summary_output_dir}")

Summaries directory: /Users/lauren/repos/PAD2Skills/data/silver/pad_summaries
Short summary output directory: /Users/lauren/repos/PAD2Skills/data/silver/short_summary_json


In [28]:
# Generate short summaries for each selected project
for project_id in selected_projects:
    # Check if summary exists for this project
    summary_file = summaries_output_dir / f"{project_id}_summary.txt"
    
    if not summary_file.exists():
        print(f"⚠ No summary found: {project_id}")
        continue
    
    # Generate short summary using the src utility
    results = generate_all_short_summaries(
        summaries_dir=summaries_output_dir,
        output_dir=short_summary_output_dir,
        specific_project=project_id,
        overwrite=ow_7_04_short_summary  # Reuse the same overwrite flag as regular summaries
    )
    
    # Report result
    if results["generated"]:
        print(f"✓ Generated short summary: {project_id}")
    elif results["skipped"]:
        print(f"○ Skipped (already exists): {project_id}")
    elif results["failed"]:
        error = results["failed"][0][1]
        print(f"✗ Failed: {project_id} - {error}")

○ Skipped (already exists): P119893
○ Skipped (already exists): P173506
○ Skipped (already exists): P176731
○ Skipped (already exists): P507759
○ Skipped (already exists): P180547
○ Skipped (already exists): P505856
○ Skipped (already exists): P181341
○ Skipped (already exists): P075941
○ Skipped (already exists): P160708
○ Skipped (already exists): P153743


2026-01-14 14:24:33,045 - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


✓ Generated short summary: P166796


2026-01-14 14:24:44,086 - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


✓ Generated short summary: P175295


...

✓ Generated short summary: P149683


2026-01-14 14:32:51,733 - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


✓ Generated short summary: P181221


2026-01-14 14:32:56,921 - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


✓ Generated short summary: P163752


## 8. Pipeline Step #6: Extract Occupations and Skills

### 8.01 Import Occupations Extraction Module

In [27]:
from src.extraction.occupations_extractor import extract_all_occupations

print("✓ Occupations extraction module imported")

✓ Occupations extraction module imported


### 8.02 Set Up Occupations Extraction Paths

In [28]:
# Set up paths for occupations extraction
occupations_output_dir = project_root / "data" / "silver" / "occupations_skills_json"

print(f"Chunks directory: {chunks_output_dir}")
print(f"Abbreviations directory: {abbreviations_output_dir}")
print(f"Summaries directory: {summaries_output_dir}")
print(f"Occupations output directory: {occupations_output_dir}")

Chunks directory: /Users/lauren/repos/PAD2Skills/data/silver/pads_md_chunks
Abbreviations directory: /Users/lauren/repos/PAD2Skills/data/silver/abbreviations_md
Summaries directory: /Users/lauren/repos/PAD2Skills/data/silver/pad_summaries
Occupations output directory: /Users/lauren/repos/PAD2Skills/data/silver/occupations_skills_json


### 8.03 Extract Occupations for Selected Projects

In [29]:
# Extract occupations for each selected project
for project_id in selected_projects:
    # Check if chunks exist for this project
    chunk_files = list(chunks_output_dir.glob(f"{project_id}_*.md"))
    
    if not chunk_files:
        print(f"⚠ No chunks found: {project_id}")
        continue
    
    # Extract occupations using the src utility
    results = extract_all_occupations(
        chunks_dir=chunks_output_dir,
        output_dir=occupations_output_dir,
        abbr_dir=abbreviations_output_dir,
        summary_dir=summaries_output_dir,
        specific_project=project_id,
        overwrite=ow_8_03_extract_occupations
    )
    
    # Report result
    if results["generated"]:
        occupation_files = list(occupations_output_dir.glob(f"{project_id}_*_occupations.json"))
        print(f"✓ Extracted occupations: {project_id} ({len(results['generated'])} files)")
    elif results["skipped"]:
        print(f"○ Skipped (already exists): {project_id}")
    elif results["failed"]:
        error = results["failed"][0][1] if results["failed"] else "Unknown error"
        print(f"✗ Failed: {project_id} - {error}")

○ Skipped (already exists): P119893
○ Skipped (already exists): P173506
○ Skipped (already exists): P176731
○ Skipped (already exists): P507759
○ Skipped (already exists): P180547
○ Skipped (already exists): P505856
○ Skipped (already exists): P181341
○ Skipped (already exists): P075941
○ Skipped (already exists): P160708
○ Skipped (already exists): P153743
○ Skipped (already exists): P166796
○ Skipped (already exists): P175295
○ Skipped (already exists): P144135
○ Skipped (already exists): P164225
○ Skipped (already exists): P179797
○ Skipped (already exists): P164044
○ Skipped (already exists): P503941
○ Skipped (already exists): P179380
○ Skipped (already exists): P167569
○ Skipped (already exists): P168185
○ Skipped (already exists): P171742
○ Skipped (already exists): P166170
○ Skipped (already exists): P164885
○ Skipped (already exists): P180575
○ Skipped (already exists): P178914
○ Skipped (already exists): P173749
○ Skipped (already exists): P120304
○ Skipped (already exists): 

### 8.04 (Optional) Prepare PAD Occupations CSV

For inspection and debugging, prepare CSV files from the occupation JSON extractions. These create intermediate CSV files with flattened extractions and combined text fields.

**Note:** These CSV files are optional and not required for the production matching workflow (Step 9), which reads directly from JSON files.

In [30]:
from src.extraction.occupations_extractor import prepare_pad_occupations_csv

print("✓ PAD occupations CSV preparation module imported")

✓ PAD occupations CSV preparation module imported


In [31]:
# Set up paths for CSV preparation
occupation_skills_csv_dir = project_root / "data" / "silver" / "occupation_skills_csv"

print(f"Occupation JSON input: {occupations_output_dir}")
print(f"CSV output directory: {occupation_skills_csv_dir}")

Occupation JSON input: /Users/lauren/repos/PAD2Skills/data/silver/occupations_skills_json
CSV output directory: /Users/lauren/repos/PAD2Skills/data/silver/occupation_skills_csv


In [32]:
# Prepare CSV files for each selected project
for project_id in selected_projects:
    # Check if occupation JSON files exist for this project
    occupation_files = list(occupations_output_dir.glob(f"{project_id}_*_occupations.json"))
    
    if not occupation_files:
        print(f"⚠ No occupation files found: {project_id}")
        continue
    
    # Prepare CSV using the src utility
    try:
        results = prepare_pad_occupations_csv(
            json_dir=occupations_output_dir,
            output_dir=occupation_skills_csv_dir,
            specific_project=project_id,
            overwrite=ow_8_04_extract_occs_csv
        )
        
        # Report result
        if results["generated"]:
            csv_file = occupation_skills_csv_dir / f"{project_id}_pad_occupations_prepared.csv"
            if csv_file.exists():
                # Count rows in the CSV
                import pandas as pd
                df = pd.read_csv(csv_file)
                print(f"✓ Prepared CSV: {project_id} ({len(df):,} extractions)")
            else:
                print(f"✓ Prepared CSV: {project_id}")
        elif results["skipped"]:
            print(f"○ Skipped (already exists): {project_id}")
        elif results["failed"]:
            error = results["failed"][0][1] if results["failed"] else "Unknown error"
            print(f"✗ Failed: {project_id} - {error}")
    except Exception as e:
        print(f"✗ Failed: {project_id} - {e}")

○ Skipped (already exists): P119893
○ Skipped (already exists): P173506
○ Skipped (already exists): P176731
○ Skipped (already exists): P507759
○ Skipped (already exists): P180547
○ Skipped (already exists): P505856
○ Skipped (already exists): P181341
○ Skipped (already exists): P075941
○ Skipped (already exists): P160708
○ Skipped (already exists): P153743
○ Skipped (already exists): P166796
○ Skipped (already exists): P175295
○ Skipped (already exists): P144135
○ Skipped (already exists): P164225
○ Skipped (already exists): P179797
○ Skipped (already exists): P164044
○ Skipped (already exists): P503941
○ Skipped (already exists): P179380
○ Skipped (already exists): P167569
○ Skipped (already exists): P168185
○ Skipped (already exists): P171742
○ Skipped (already exists): P166170
○ Skipped (already exists): P164885
○ Skipped (already exists): P180575
○ Skipped (already exists): P178914
○ Skipped (already exists): P173749
○ Skipped (already exists): P120304
○ Skipped (already exists): 

## 9. Pipeline Step #7: Match PAD Occupations to ESCO

### 9.01 Import ESCO Matching Modules

In [33]:
from src.matching.esco_prepare import prepare_esco_data
from src.matching.pad_matcher import match_pad_to_esco

print("✓ ESCO matching modules imported")

✓ ESCO matching modules imported


### 9.02 Prepare ESCO Data (Run Once)

In [34]:
# Prepare ESCO data with embeddings (only needs to be run once)
esco_csv = project_root / "data" / "bronze" / "esco" / "occupations_en.csv"
esco_relations_csv = project_root / "data" / "bronze" / "esco" / "occupationSkillRelations_en.csv"
esco_output_csv = project_root / "data" / "silver" / "clean_esco" / "esco_occupations_prepared.csv"
esco_embeddings_file = project_root / "data" / "silver" / "embeddings" / "esco_embeddings.npy"

# Check if ESCO data is already prepared
if esco_output_csv.exists() and esco_embeddings_file.exists():
    print("○ ESCO data already prepared (skipping)")
else:
    print("Preparing ESCO data with embeddings...")
    prepare_esco_data(
        esco_csv=esco_csv,
        esco_relations_csv=esco_relations_csv,
        output_csv=esco_output_csv,
        embeddings_file=esco_embeddings_file,
        model_name="intfloat/e5-small-v2",
        overwrite_embeddings=ow_9_02_esco_embeddings
    )
    print("✓ ESCO data prepared successfully")

○ ESCO data already prepared (skipping)


### 9.03 Set Up PAD Matching Paths

In [35]:
# Set up paths for PAD matching
pad_occupations_dir = project_root / "data" / "silver" / "occupations_skills_json"
esco_matching_csv_dir = project_root / "data" / "silver" / "esco_matching_csv"
esco_matching_json_dir = project_root / "data" / "silver" / "esco_matching_json"

print(f"PAD occupations directory: {pad_occupations_dir}")
print(f"ESCO prepared CSV: {esco_output_csv}")
print(f"ESCO embeddings: {esco_embeddings_file}")
print(f"Matching CSV output: {esco_matching_csv_dir}")
print(f"Matching JSON output: {esco_matching_json_dir}")

PAD occupations directory: /Users/lauren/repos/PAD2Skills/data/silver/occupations_skills_json
ESCO prepared CSV: /Users/lauren/repos/PAD2Skills/data/silver/clean_esco/esco_occupations_prepared.csv
ESCO embeddings: /Users/lauren/repos/PAD2Skills/data/silver/embeddings/esco_embeddings.npy
Matching CSV output: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_csv
Matching JSON output: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_json


### 9.04 Match PAD Occupations to ESCO for Selected Projects

In [36]:
# Set overwrite parameter
overwrite_matching = False  # Set to True to force re-matching

# Match PAD occupations to ESCO for each selected project
for project_id in selected_projects:
    # Check if occupation JSON files exist for this project
    occupation_files = list(pad_occupations_dir.glob(f"{project_id}_*.json"))
    
    if not occupation_files:
        print(f"⚠ No occupation files found: {project_id}")
        continue
    
    # Match PAD occupations to ESCO
    try:
        match_pad_to_esco(
            pad_occupations_dir=pad_occupations_dir,
            project_id=project_id,
            esco_csv=esco_output_csv,
            esco_embeddings=esco_embeddings_file,
            output_dir=project_root / "data" / "silver",
            model_name="intfloat/e5-small-v2",
            top_k=20,
            chunk_size=75,
            save_diagnostics=True,
            overwrite=overwrite_matching
        )
        print(f"✓ Matched occupations: {project_id}")
    except Exception as e:
        print(f"✗ Failed: {project_id} - {e}")

Output already exists: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_csv/P119893_esco_matches.csv
Use overwrite=True to force re-matching
✓ Matched occupations: P119893
Output already exists: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_csv/P173506_esco_matches.csv
Use overwrite=True to force re-matching
✓ Matched occupations: P173506
Output already exists: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_csv/P176731_esco_matches.csv
Use overwrite=True to force re-matching
✓ Matched occupations: P176731
Output already exists: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_csv/P507759_esco_matches.csv
Use overwrite=True to force re-matching
✓ Matched occupations: P507759
Output already exists: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_csv/P180547_esco_matches.csv
Use overwrite=True to force re-matching
✓ Matched occupations: P180547
Output already exists: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_csv/P505856_esco_mat

## 10. Pipeline Step #8: Select Best ESCO Match

(7a) Use OpenAI API to select the single best ESCO occupation match for each PAD occupation from the candidates identified in Step 6. (7b) Then create a file unique on the ESCO matches.

### 10.01 Import ESCO Selector Function

In [37]:
from src.matching.esco_selector import select_best_esco_matches

print("✓ ESCO selector function imported")

✓ ESCO selector function imported


### 10.02 Set Up ESCO Selection Paths

In [38]:
# Set up paths for ESCO selection
selection_json_dir = project_root / "data" / "silver" / "choose_esco_json"
selection_csv_dir = project_root / "data" / "silver" / "choose_esco_csv"

print(f"ESCO matching JSON input: {esco_matching_json_dir}")
print(f"Selection JSON output: {selection_json_dir}")
print(f"Selection CSV output: {selection_csv_dir}")

ESCO matching JSON input: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_json
Selection JSON output: /Users/lauren/repos/PAD2Skills/data/silver/choose_esco_json
Selection CSV output: /Users/lauren/repos/PAD2Skills/data/silver/choose_esco_csv


### 10.03 Select Best ESCO Matches for Selected Projects

In [39]:
# Select best ESCO matches for each selected project
for project_id in selected_projects:
    # Check if ESCO matching JSON files exist for this project
    matching_files = list(esco_matching_json_dir.glob(f"{project_id}_*_esco_matches.json"))
    
    if not matching_files:
        print(f"⚠ No ESCO matching files found: {project_id}")
        continue
    
    # Select best ESCO matches
    try:
        df = select_best_esco_matches(
            input_dir=esco_matching_json_dir,
            project_id=project_id,
            pad_occupations_dir=occupations_output_dir,
            output_json_dir=selection_json_dir,
            output_csv_dir=selection_csv_dir,
            overwrite=ow_10_03_esco_selection
        )
        
        # Print summary statistics
        total = len(df)
        selected = df['esco_id'].notna().sum()
        needs_review = df['needs_manual_review'].sum()
        
        print(f"✓ Selected ESCO matches: {project_id}")
        print(f"  Total records: {total:,}")
        print(f"  ESCO matches selected: {selected:,}")
        print(f"  Needs manual review: {needs_review:,}")
        print()
        
    except Exception as e:
        print(f"✗ Failed: {project_id} - {e}")
        print()


Found 2 JSON chunk files for project P119893
[1/2] Skipping existing: P119893_000-074_esco_selection.json
[2/2] Skipping existing: P119893_075-093_esco_selection.json
✓ Processed 2 chunks
✓ Results saved to: /Users/lauren/repos/PAD2Skills/data/silver/choose_esco_json
✓ Loaded original PAD data: 94 rows
Loading 2 JSON selection files...
✓ Combined 94 records from 2 files
✓ Saved combined selections to: /Users/lauren/repos/PAD2Skills/data/silver/choose_esco_csv/P119893_esco_selections.csv
  Rows: 94, Columns: 12
✓ Selected ESCO matches: P119893
  Total records: 94
  ESCO matches selected: 94
  Needs manual review: 0

Found 2 JSON chunk files for project P173506
[1/2] Skipping existing: P173506_000-074_esco_selection.json
[2/2] Skipping existing: P173506_075-080_esco_selection.json
✓ Processed 2 chunks
✓ Results saved to: /Users/lauren/repos/PAD2Skills/data/silver/choose_esco_json
✓ Loaded original PAD data: 81 rows
Loading 2 JSON selection files...
✓ Combined 81 records from 2 files
✓ Sa

### 10.04 Import Unique ESCO Function

In [40]:
from src.matching.unique_esco import create_unique_esco_matches

print("✓ Unique ESCO function imported")

✓ Unique ESCO function imported


### 10.05 Set Up Unique ESCO Paths

In [41]:
# Set up paths for unique ESCO matches
unique_esco_output_dir = project_root / "data" / "silver" / "unique_esco_csv"
esco_occupations_path = project_root / "data" / "bronze" / "esco" / "occupations_en.csv"

print(f"ESCO selections input: {selection_csv_dir}")
print(f"ESCO occupations: {esco_occupations_path}")
print(f"Unique ESCO output: {unique_esco_output_dir}")

ESCO selections input: /Users/lauren/repos/PAD2Skills/data/silver/choose_esco_csv
ESCO occupations: /Users/lauren/repos/PAD2Skills/data/bronze/esco/occupations_en.csv
Unique ESCO output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_csv


### 10.06 Create Unique ESCO Matches for Selected Projects

In [42]:
# Create unique ESCO matches for each selected project
for project_id in selected_projects:
    # Check if ESCO selections CSV exists for this project
    selections_csv_path = selection_csv_dir / f"{project_id}_esco_selections.csv"
    
    if not selections_csv_path.exists():
        print(f"⚠ No ESCO selections CSV found: {project_id}")
        continue
    
    # Set up path for sections JSON (with _1 suffix)
    sections_json_path = project_root / "data" / "silver" / "document_sections" / f"{project_id}_1_sections.json"
    
    # Set up output path
    output_path = unique_esco_output_dir / f"{project_id}_unique_matched.csv"
    
    # Create unique ESCO matches
    try:
        df_unique = create_unique_esco_matches(
            project_id=project_id,
            selections_csv_path=selections_csv_path,
            esco_occupations_path=esco_occupations_path,
            sections_json_path=sections_json_path,
            output_path=output_path,
            overwrite=ow_10_06_unique_esco
        )
        
        print(f"✓ Created unique ESCO matches: {project_id}")
        print(f"  Unique ESCO occupations: {len(df_unique):,}")
        print()
        
    except Exception as e:
        print(f"✗ Failed: {project_id} - {e}")
        print()


○ Skipped (already exists): P119893_unique_matched.csv
✓ Created unique ESCO matches: P119893
  Unique ESCO occupations: 44

○ Skipped (already exists): P173506_unique_matched.csv
✓ Created unique ESCO matches: P173506
  Unique ESCO occupations: 46

○ Skipped (already exists): P176731_unique_matched.csv
✓ Created unique ESCO matches: P176731
  Unique ESCO occupations: 46

○ Skipped (already exists): P507759_unique_matched.csv
✓ Created unique ESCO matches: P507759
  Unique ESCO occupations: 36

○ Skipped (already exists): P180547_unique_matched.csv
✓ Created unique ESCO matches: P180547
  Unique ESCO occupations: 67

○ Skipped (already exists): P505856_unique_matched.csv
✓ Created unique ESCO matches: P505856
  Unique ESCO occupations: 39

○ Skipped (already exists): P181341_unique_matched.csv
✓ Created unique ESCO matches: P181341
  Unique ESCO occupations: 70

○ Skipped (already exists): P075941_unique_matched.csv
✓ Created unique ESCO matches: P075941
  Unique ESCO occupations: 61



## 11. Pipeline Step #9: Add NACE Industry Codes

Enrich ESCO occupation matches with NACE (Statistical Classification of Economic Activities) industry codes using semantic similarity.

### 11.01 Check if ESCO-NACE Groups Exist

In [43]:
# Check if ESCO-NACE groups mapping exists
esco_nace_groups_path = project_root / "data" / "silver" / "esco_nace_csv" / "esco_nace_groups.csv"

if not esco_nace_groups_path.exists():
    print("⚠ ESCO-NACE groups file not found. Creating it now...")
    print()
    create_esco_nace = True
else:
    print(f"✓ ESCO-NACE groups file exists: {esco_nace_groups_path}")
    create_esco_nace = False

✓ ESCO-NACE groups file exists: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_csv/esco_nace_groups.csv


### 11.02 Create ESCO-NACE Groups (if needed)

In [44]:
if create_esco_nace:
    from src.nace.esco_nace_mapper import ESCONACEMapper
    
    # Set up paths
    nace_rdf_path = project_root / "data" / "bronze" / "nace" / "NACE_Rev.2.1.rdf"
    esco_occupations_path = project_root / "data" / "bronze" / "esco" / "occupations_en.csv"
    esco_nace_output_dir = project_root / "data" / "silver" / "esco_nace_csv"
    
    # Create mapper
    mapper = ESCONACEMapper(nace_rdf_path, esco_occupations_path)
    
    # Run the mapping process
    main_output, inspect_output = mapper.run(esco_nace_output_dir)
    
    print(f"\n✓ ESCO-NACE groups created")
    print(f"  Main output: {main_output}")
    print(f"  Inspection output: {inspect_output}")
else:
    print("✓ Using existing ESCO-NACE groups file")

✓ Using existing ESCO-NACE groups file


### 11.03 Import NACE Selector

In [45]:
from src.nace.nace_selector import NACESelector

print("✓ NACE selector imported")

✓ NACE selector imported


### 11.04 Set Up NACE Selection Paths

In [46]:
# Set up paths for NACE selection
unique_esco_nace_output_dir = project_root / "data" / "silver" / "unique_esco_nace_csv"

print(f"Unique ESCO input: {unique_esco_output_dir}")
print(f"ESCO-NACE groups: {esco_nace_groups_path}")
print(f"Unique ESCO + NACE output: {unique_esco_nace_output_dir}")

Unique ESCO input: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_csv
ESCO-NACE groups: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_csv/esco_nace_groups.csv
Unique ESCO + NACE output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv


### 11.05 Select Best NACE Group for Each ESCO Occupation

In [47]:
# Set overwrite parameter
overwrite_nace = True  # Set to True to force re-creation

# Select best NACE group for each selected project
for project_id in selected_projects:
    # Check if unique ESCO matches exist for this project
    unique_esco_path = unique_esco_output_dir / f"{project_id}_unique_matched.csv"
    
    if not unique_esco_path.exists():
        print(f"⚠ No unique ESCO file found: {project_id}")
        continue
    
    # Create NACE selector
    try:
        selector = NACESelector(
            unique_esco_path=unique_esco_path,
            esco_nace_groups_path=esco_nace_groups_path,
            model_name="intfloat/e5-small-v2"
        )
        
        # Run the selection process
        output_file = selector.run(
            output_dir=unique_esco_nace_output_dir,
            project_id=project_id,
            overwrite=ow_11_05_match_nace
        )
        
        print(f"✓ Added NACE codes: {project_id}")
        print(f"  Output: {output_file}")
        print()
        
    except Exception as e:
        print(f"✗ Failed: {project_id} - {e}")
        print()

○ Skipped (already exists): P119893_unique_matched_with_nace.csv
✓ Added NACE codes: P119893
  Output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv/P119893_unique_matched_with_nace.csv

○ Skipped (already exists): P173506_unique_matched_with_nace.csv
✓ Added NACE codes: P173506
  Output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv/P173506_unique_matched_with_nace.csv

○ Skipped (already exists): P176731_unique_matched_with_nace.csv
✓ Added NACE codes: P176731
  Output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv/P176731_unique_matched_with_nace.csv

○ Skipped (already exists): P507759_unique_matched_with_nace.csv
✓ Added NACE codes: P507759
  Output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv/P507759_unique_matched_with_nace.csv

○ Skipped (already exists): P180547_unique_matched_with_nace.csv
✓ Added NACE codes: P180547
  Output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv/P180547_uniqu

## 12. Pipeline Step #10: Refine ESCO Skills

Evaluate ESCO skills for relevance to PAD project context using OpenAI API. This step identifies which skills are relevant and marks the top 5 most important skills for each occupation.

### 12.01 Import Skills Refiner

In [48]:
from src.skills.skills_refiner import SkillsRefiner

print("✓ Skills refiner imported")

✓ Skills refiner imported


### 12.02 Set Up Skills Refinement Paths

In [49]:
# Set up paths for skills refinement
esco_skills_path = project_root / "data" / "bronze" / "esco" / "occupationSkillRelations_en.csv"
pad_summaries_dir = project_root / "data" / "silver" / "pad_summaries"
skills_output_dir = project_root / "data" / "silver" / "esco_nace_w_skills_csv"

print(f"Unique ESCO + NACE input: {unique_esco_nace_output_dir}")
print(f"ESCO skills file: {esco_skills_path}")
print(f"PAD summaries dir: {pad_summaries_dir}")
print(f"Skills output: {skills_output_dir}")

Unique ESCO + NACE input: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv
ESCO skills file: /Users/lauren/repos/PAD2Skills/data/bronze/esco/occupationSkillRelations_en.csv
PAD summaries dir: /Users/lauren/repos/PAD2Skills/data/silver/pad_summaries
Skills output: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_w_skills_csv


### 12.03 Refine Skills for Each Project

In [50]:
# Refine skills for each selected project
for project_id in selected_projects:
    # Check if required files exist for this project
    unique_esco_nace_path = unique_esco_nace_output_dir / f"{project_id}_unique_matched_with_nace.csv"
    pad_summary_path = pad_summaries_dir / f"{project_id}_summary.txt"
    
    if not unique_esco_nace_path.exists():
        print(f"⚠ No unique ESCO with NACE file found: {project_id}")
        continue
    
    if not pad_summary_path.exists():
        print(f"⚠ No PAD summary found: {project_id}")
        continue
    
    # Create skills refiner
    try:
        refiner = SkillsRefiner(
            unique_esco_nace_file=unique_esco_nace_path,
            esco_skills_file=esco_skills_path,
            pad_summary_file=pad_summary_path,
            project_id=project_id,
            openai_api_key=OPENAI_API_KEY,
            chunk_size=3  # 3 occupations per API call
        )
        
        # Run the refinement process
        output_file = refiner.run(
            output_dir=skills_output_dir,
            overwrite=ow_12_03_refine_skills
        )
        
        print(f"✓ Refined skills: {project_id}")
        print(f"  Output: {output_file}")
        print()
        
    except Exception as e:
        print(f"✗ Failed: {project_id} - {e}")
        print()

○ Skipped (already exists): P119893_esco_nace_with_skills.csv
✓ Refined skills: P119893
  Output: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_w_skills_csv/P119893_esco_nace_with_skills.csv



○ Skipped (already exists): P173506_esco_nace_with_skills.csv
✓ Refined skills: P173506
  Output: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_w_skills_csv/P173506_esco_nace_with_skills.csv

○ Skipped (already exists): P176731_esco_nace_with_skills.csv
✓ Refined skills: P176731
  Output: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_w_skills_csv/P176731_esco_nace_with_skills.csv

○ Skipped (already exists): P507759_esco_nace_with_skills.csv
✓ Refined skills: P507759
  Output: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_w_skills_csv/P507759_esco_nace_with_skills.csv

○ Skipped (already exists): P180547_esco_nace_with_skills.csv
✓ Refined skills: P180547
  Output: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_w_skills_csv/P180547_esco_nace_with_skills.csv

○ Skipped (already exists): P505856_esco_nace_with_skills.csv
✓ Refined skills: P505856
  Output: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_w_skills_csv/P505856_esco_nace_with_skills.csv

○ Ski

## 13. Pipeline Step #11: Add O*NET Education Levels

Enrich ESCO occupation data with O*NET job zones (education and experience requirements). This includes creating the ESCO-ONET crosswalk with LLM-generated substitutions for missing values, and merging job zones onto project files.

### 13.01 Import O*NET Classes

In [51]:
from src.onet.onet_crosswalk import OnetCrosswalkCreator
from src.onet.onet_merger import OnetMerger

print("✓ O*NET classes imported")

✓ O*NET classes imported


### 13.02 Set Up O*NET Crosswalk Paths

In [52]:
# Set up paths for O*NET crosswalk creation
onet_crosswalk_path = project_root / "data" / "bronze" / "onet" / "esco_onet_crosswalk.csv"
onet_job_zones_path = project_root / "data" / "bronze" / "onet" / "onet_job_zones.txt"
esco_prepared_path = project_root / "data" / "silver" / "clean_esco" / "esco_occupations_prepared.csv"
esco_onet_output_path = project_root / "data" / "silver" / "clean_esco" / "esco_onet_job_zones.csv"

print(f"ESCO-O*NET crosswalk: {onet_crosswalk_path}")
print(f"O*NET job zones: {onet_job_zones_path}")
print(f"ESCO prepared: {esco_prepared_path}")
print(f"Output: {esco_onet_output_path}")

ESCO-O*NET crosswalk: /Users/lauren/repos/PAD2Skills/data/bronze/onet/esco_onet_crosswalk.csv
O*NET job zones: /Users/lauren/repos/PAD2Skills/data/bronze/onet/onet_job_zones.txt
ESCO prepared: /Users/lauren/repos/PAD2Skills/data/silver/clean_esco/esco_occupations_prepared.csv
Output: /Users/lauren/repos/PAD2Skills/data/silver/clean_esco/esco_onet_job_zones.csv


### 13.03 Create ESCO-O*NET Crosswalk with Job Zones (Run Once)

Create comprehensive ESCO-O*NET job zones mapping by merging O*NET crosswalk data with job zones and using LLM to estimate job zones for missing ESCO occupations.

In [53]:
# Initialize OpenAI client
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

# Create the ESCO-O*NET job zones crosswalk
crosswalk_creator = OnetCrosswalkCreator(client)

output_file = crosswalk_creator.create_crosswalk(
    crosswalk_file=onet_crosswalk_path,
    job_zones_file=onet_job_zones_path,
    esco_prepared_file=esco_prepared_path,
    output_file=esco_onet_output_path,
    chunk_size=50,
    overwrite=ow_13_03_create_crosswalk
)

print(f"✓ ESCO-O*NET crosswalk created: {output_file}")

Output file already exists: /Users/lauren/repos/PAD2Skills/data/silver/clean_esco/esco_onet_job_zones.csv
Use --overwrite to force recreation
✓ ESCO-O*NET crosswalk created: /Users/lauren/repos/PAD2Skills/data/silver/clean_esco/esco_onet_job_zones.csv


### 13.04 Set Up O*NET Merger Paths

In [54]:
# Set up paths for O*NET merger
onet_input_dir = unique_esco_nace_output_dir  # Use unique ESCO-NACE files as input
onet_output_dir = project_root / "data" / "silver" / "unique_esco_nace_onet_csv"

print(f"Job zones crosswalk: {esco_onet_output_path}")
print(f"Input dir (ESCO-NACE): {onet_input_dir}")
print(f"Output dir (ESCO-NACE-O*NET): {onet_output_dir}")

Job zones crosswalk: /Users/lauren/repos/PAD2Skills/data/silver/clean_esco/esco_onet_job_zones.csv
Input dir (ESCO-NACE): /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv
Output dir (ESCO-NACE-O*NET): /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_onet_csv


### 13.05 Merge O*NET Job Zones onto Project Files

Merge O*NET job zones onto each project's unique ESCO-NACE file, adding education and experience requirements.

In [55]:
# Merge O*NET job zones for each selected project
merger = OnetMerger()

for project_id in selected_projects:
    # Check if required input file exists for this project
    input_file = onet_input_dir / f"{project_id}_unique_matched_with_nace.csv"
    
    if not input_file.exists():
        print(f"⚠ No unique ESCO with NACE file found: {project_id}")
        continue
    
    # Set up output file path
    output_file = onet_output_dir / f"{project_id}_esco_nace_onet.csv"
    
    # Run the merge
    try:
        result_file = merger.merge_job_zones_to_project(
            job_zones_file=esco_onet_output_path,
            project_esco_nace_file=input_file,
            output_file=output_file,
            overwrite=ow_13_05_merge_onet
        )
        
        print(f"✓ Merged O*NET job zones: {project_id}")
        print(f"  Output: {result_file}")
        print()
        
    except Exception as e:
        print(f"✗ Failed: {project_id} - {e}")
        print()

Output file already exists: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_onet_csv/P119893_esco_nace_onet.csv
Use --overwrite to force recreation
✓ Merged O*NET job zones: P119893
  Output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_onet_csv/P119893_esco_nace_onet.csv

Output file already exists: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_onet_csv/P173506_esco_nace_onet.csv
Use --overwrite to force recreation
✓ Merged O*NET job zones: P173506
  Output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_onet_csv/P173506_esco_nace_onet.csv

Output file already exists: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_onet_csv/P176731_esco_nace_onet.csv
Use --overwrite to force recreation
✓ Merged O*NET job zones: P176731
  Output: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_onet_csv/P176731_esco_nace_onet.csv

Output file already exists: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_onet_csv/